# Õppetund 18: AI-agentide turvamine krüptograafiliste kviitungitega

## Praktiline märkmik

See märkmik juhendab läbi nelja ülesande:

1. **Allkirjasta oma esimene kviitung** agendi tööriista kutse jaoks ja kontrolli see üle.
2. **Muuda kviitungit** ja vaata, kuidas kontroll ebaõnnestub.
3. **Loo kolm-kviitungiline kett** ja kinnita keti terviklikkus.
4. **Pöörle Microsoft Agent Framework tööriista kutse ümber**, nii et iga tegevus tekitab kviitungi.

Kõik krüptograafilised primitiivid imporditakse hästi hooldatud teekidest (`pynacl` Ed25519 jaoks, `jcs` RFC 8785 kanonilise JSONi jaoks, `hashlib` Python standardteegist SHA-256 jaoks). Kviitungiloogika ise on lihtne Python, mida saad lugeda ja muuta.

Käivita lahtrid järjest. Iga osa on lühike ja iseseisev.


## Seadistus

Paigaldage need kaks sõltuvust. Mõlemal on lubavad litsentsid (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Abivahendid

Need kaks abivahendit tegelevad base64url kodeerimisega (ilma täiendava täitmiseta) ja SHA-256 räsi loomisega suvalistest objektidest. Need hoiavad ülejäänud märkmiku keskendununa alles kviitungi loogikale.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## 1. peatükk: Allkirjasta oma esimene tšekk

Kujuta ette, et meie esindaja ettevõttest **Contoso Travel** otsis kliendi jaoks lennupileteid Sydney ja Los Angelese vahel. Tahame selle tööriista kasutamise salvestada allkirjastatud tšekina, et tulevikus audiitor saaks seda kontrollida ilma, et ta peaks meid usaldama.

### Samm 1.1: Genereeri allkirjastamisvõti

Tootmiskeskkonnas elab agendi allkirjastamisvõti riistvaralises turvamoodulis (HSM), Azure Key Vaultis või mõnes sarnases kaitstud hoidlas. Selle õppetunni jaoks genereerime uue võtme mällu.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Samm 1.2: Koosta kviitungi koormus

Koormus sisaldab kõike, millele tahame kviitungiga kinnitust: kes tegutses, mis tööriistaga, milliste argumentidega, mis tuli tagasi, millise poliisi alusel ja millal. Me räsiarvutame argumendid ja tulemuse selle asemel, et need kas otse kviitungisse lisada, et kviitung ei lekkiks tundlikku sisu.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Samm 1.3: Digitaalselt allkirjasta ja koosta kviitung

Kolm sammu:

1. Canonicalize the payload using JCS so two implementations producing the same logical receipt produce byte-identical bytes.
2. Räsi kanonilisi baitide SHA-256-ga.
3. Allkirjasta räsi Ed25519 privaatvõtmega.

Allkiri kinnitatakse seejärel algse andmepaketile, et toota lõplik kviitung.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Samm 1.4: Kontrolli kviitungit

Kontrollimine on protsessi tagurpidi sõlmimine. Eemaldame allkirja, arvutame kanonilise räsi uuesti välja ja kontrollime allkirja kviitungis oleva avaliku võtmega.

Auditeerija, kes seda kontrolli teeb, ei vaja meilt muud kui ainult kviitungit ennast. Ei mingit teenusekutset, ei mingit võtmedirektori päringut, ei mingit usaldust.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Peaksite nägema `Receipt is valid: True`. Esindaja on loonud oma esimese krüptograafiliselt allkirjastatud auditi kande.


## 2. peatükk: Kviitungi eest kättemaksmine

Kviitungide põhieesmärk on see, et neid ei saa varjatult muuta. Tõestame seda.

Muudame täpselt ühe tähe kviitungis ja vaatame, kuidas kontroll ei õnnestu.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Mis just juhtus?

Kui me muutsime `policy_id`, muutusid ka kanoonilised baidid. Nende baitide SHA-256 räsi muutus. Allkiri (mis oli algse räsi peal) ei klapi enam uue räsi väärtusega. Kontrollimine tagastab õigesti `False`.

Pole võimalik ühtki tšeki välja muuta ja siiski see valideeruks, kui ründajal pole privaatvõtit. Niikaua kui privaatvõti asub võtmekapil ja avalik võti on avaldatud, on manipuleerimist võimatu varjata.

Proovi ise: muuda kõrgemal lahtris `tool_name` või `agent_id` või `timestamp` ja käivita uuesti. Iga muudatus tekitab kehtetu tšeki.


## 3. peatükk: Kviitungeid omavahel keti kaudu ühendada

Üks kviitung kaitseb ühte toimingut. Enamik agente sooritab palju toiminguid. Et kogu järjestust võltsimiskindlaks muuta, ühendame iga kviitungi eelmise kviitungiga, lisades uue kviitungi andmepaketisse eelmise kviitungi räsi.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Kui keegi eemaldab või ümber järjestab kviitungi, katkeb kett täpselt sellel kohal. Iga hilisema kviitungi kontroll ebaõnnestub, sest selle `previous_receipt_hash` ei vasta enam tegeliku eelkäija räsidele.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Nüüd katkesta ahel, manipuleerides keskmise kviitungiga ja kontrolli uuesti. Manipuleeritud kviitung ei läbi allkirjakontrolli, JA järgmine kviitung ebaõnnestub ahelalingi kontrollis (sest selle `previous_receipt_hash` ei vasta enam muudetud keskmise kviitungi räsi väärtusele).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Kviitung 0 on endiselt kehtiv (seda ei muudetud ega ole sõltuvusi eelkäijast). Kviitung 1 ebaõnnestub allkirjakontrollis, sest me muutsime `tool_args_hash`. Kviitung 2 ebaõnnestub oma ahela lingi kontrollis, sest selle `previous_receipt_hash` arvutati algse (nüüd muudetud) kviitungi 1 põhjal.

Isegi kui ründaja uuesti allkirjastab muudetud kviitungi 1 (mida ta ei saa teha ilma privaatvõtmeta), paljastaks kviitungi 2 ahela lingi mittevastavus siiski võltsimise. Muutuse varjamiseks peaks ründaja uuesti allkirjastama iga kviitungi alates muudatuse punktist, mis nõuab privaatvõtme omamist.


## Jaotis 4: Kättetõendi allkirjastamisega agendi tööriista kutsungi mähkimine

Tootmiskeskkonnas ei soovi te, et iga agendi autor peaks meeles pidama `make_receipt` kutsumist. Soovite, et kättetõendi allkirjastamine oleks automaatne iga tööriista käivituse puhul.

Siin on lihtsaim mustrilaadne lahendus: mähise klass, mis võtab mis tahes kutsutava tööriistafunktsiooni ja tagastab selle kättetõendit väljastava versiooni. See kohandub iga agendi raamistikuga, sealhulgas Microsoft Agent Frameworkiga (`agent_framework.foundry`).

Kui teil pole Microsoft Foundry projekti üles seatud, demonstreerib allpool olev kohalik testimiskood siiski seda mustrit.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Integratsioon Microsoft Agent Frameworkiga

Ülalolev `ReceiptedTool` wrapper on raamistikust sõltumatu. Selle kasutamiseks Microsoft Agent Frameworkiga loodud agendi sees registreerige mähitud funktsioon tööriistana. Skeem (te asendaksite mocki päris Microsoft Foundry tööriista registreerimisega):

```python
# Pseudokood, mis näitab integratsiooni kuju.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Sa oled Contoso Reisiagent ...",
#     tools=[receipted_lookup],   # pakitud tööriist, mitte tavaline funktsioon
# )
# response = agent.run("Leia lennud Sydneyst Los Angelesesse juunis.")
#
# # Pärast käivitust on iga tööriista kutse, mille agent tegi, allkirjastatud kviitung:
# audit_chain = receipted_lookup.receipts
```

Agendi raamistik ei pea teadma midagi tšekkidest. Tšeki allkirjastamine on mähitud tööriista ümber, mitte raamistikku külge pandud. Sel viisil lisate päritolu olemasolevale agendi koodile ilma agenti ümber kirjutamata.


## Kordus ja lisaväljakutse

Sa tegid järgmist:

- Loodud Ed25519 võtmepar.
- Koostatud ja allkirjastatud agenttööriista kutse kviitung.
- Kviitungi võrguühenduseta avaliku võtmega kinnitatud.
- Muudetud kviitungit ja täheldatud kinnituse ebaõnnestumist.
- Koostatud kolmest kviitungist koosnev hashlinkide järjend.
- Muudetud järjendi keskmist osa ja täheldatud nii allkirja kui ka järjendi lingi rikkeid.
- Pakendatud tööriista funktsioon automaatse kviitungi allkirjastamisega.

**Lisaväljakutse.** Laienda kviitungi skeemi `request_id` väljal (UUID hajutatud jälgimiseks). Uuenda `make_receipt`, et see kaasaks selle välja, ja kinnita, et kviitungid kinnituvad lõpust lõpuni. Seejärel muuda väli pärast allkirjastamist ja kinnita, et kinnitamine ebaõnnestub. See sunnib sind sisemiselt mõistma, kuidas iga bait kanonilises kodeeringus allkirja mõjutab.

**Oluline piir.** Kviitungid tõendavad kolme asja ja ainult kolme asja: attribuut (see võti allkirjastas selle sisu), terviklikkus (sisu pole allkirjastamisest alates muutunud) ja järjekord (see kviitung tuli pärast seda kviitungit). Nad EI tõenda, et agendi tegevus oli õige, et `policy_id`-ga nimetatud poliitika tõepoolest hinnati või et agent järgis kõiki reegleid. Kviitungid on alus. Juhtimine on süsteem, mille ehitad selle peale.

Loe õppetunni README uuesti selle piiri seisukohalt. Kõige levinum viga tiimide seas kviitungitega on arvata, et "meil on kviitungid" tähendab "meid juhitakse." See ei tähenda. Kviitungid teevad agendi käitumise auditeeritavaks. Nad ei tee seda õigeks.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
